<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/15%20-%20Simulacao%20de%20Vazamentos%20e%20Desvio%20Automatico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 15 - Notebook: Simulação de Injeção de Falhas e Desvio Automático de Fluxo

Neste notebook simulamos uma contingência em tempo real: o travamento mecânico (jamming) de uma esteira principal e o recálculo dinâmico de rota pelo SCADA para direcionar as peças por uma esteira de desvio (bypass) até a caixa de loteamento.


In [1]:
import time
import heapq
from typing import List, Dict, Set, Tuple, Optional, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII alinhada."""
    if not dados:
        return "Tabela Vazia"
    cols = list(dados[0].keys())
    w = {c: max(len(c), max(len(str(r.get(c, ''))) for r in dados)) for c in cols}
    head = " | ".join(f"{c:<{w[c]}}" for c in cols)
    div = "-+-".join("-" * w[c] for c in cols)
    rows = [" | ".join(f"{str(r.get(c, '')):<{w[c]}}" for c in cols) for r in dados]
    return "\n".join([head, div] + rows)

class GrafoManufatura:
    """Representa a malha física da FMS como Dígrafo Ponderado G = (V, E, W)."""
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_rota(self, u: str, v: str, comp: float, atuador: str, mecanismo: str = "Esteira"):
        ui = self.v_to_idx[u]
        vi = self.v_to_idx[v]
        self.adj_binaria[ui][vi] = 1
        self.adj_pesos[ui][vi] = comp
        self.arestas_detalhes.append({
            "Origem": u, "Destino": v, "Comprimento (m)": comp,
            "Atuador ISA": atuador, "Mecanismo": mecanismo
        })

def criar_planta_fms_com_contingencia() -> GrafoManufatura:
    postos = [
        "SILO-100", "INSP-200", "DIV-201", "DIV-202", "DIV-203",
        "CX-301", "CX-302", "CX-303", "DESC-304"
    ]
    g = GrafoManufatura(postos)
    g.adicionar_rota("SILO-100", "INSP-200", 0.8, "XV-101", "Alimentador Pneumático")
    g.adicionar_rota("INSP-200", "DIV-201",  1.2, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-201",  "CX-301",   0.5, "XV-201", "Pistão de Triagem 1")
    g.adicionar_rota("DIV-201",  "DIV-202",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-202",  "CX-302",   0.5, "XV-202", "Pistão de Triagem 2")
    g.adicionar_rota("DIV-202",  "DIV-203",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-203",  "CX-303",   0.5, "XV-203", "Pistão de Triagem 3")
    g.adicionar_rota("DIV-203",  "DESC-304", 0.8, "M-101",  "Calha de Descarte")

    # Rotas de contingência e bypass para tolerância a falhas mecânicas
    g.adicionar_rota("INSP-200", "DIV-203",  2.5, "XV-200B", "Bypass de Contingência")
    g.adicionar_rota("DIV-201",  "DIV-203",  2.2, "M-101B",  "Contorno Esteira Secundária")
    return g

class SistemaDesvioAutomaticoFMS:
    """Controlador de resposta a contingências operacionais com reponderação dinâmica."""
    def __init__(self, grafo: GrafoManufatura):
        self.g = grafo

    def dijkstra(self, src: str, dst: str) -> Tuple[float, List[str]]:
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[src] = 0.0
        pq = [(0.0, src)]
        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]:
                continue
            if u == dst:
                break
            ui = self.g.v_to_idx[u]
            for vi in range(self.g.n):
                peso = self.g.adj_pesos[ui][vi]
                if peso < float('inf'):
                    v = self.g.idx_to_v[vi]
                    if dist[u] + peso < dist[v]:
                        dist[v] = dist[u] + peso
                        pred[v] = u
                        heapq.heappush(pq, (dist[v], v))
        path, curr = [], dst
        while curr:
            path.append(curr)
            curr = pred[curr]
        path.reverse()
        if path and path[0] == src:
            return dist[dst], path
        return float('inf'), []

    def tratar_evento_travamento(self, u_falha: str, v_falha: str,
                                  origem_lote: str, destino_lote: str) -> Dict[str, Any]:
        """Aplica punição topológica W(u, v) <- inf e recalcula rota no ciclo de varredura."""
        t0 = time.perf_counter()
        ui = self.g.v_to_idx[u_falha]
        vi = self.g.v_to_idx[v_falha]

        # Punição topológica: isolamento mecânico da esteira
        self.g.adj_pesos[ui][vi] = float('inf')
        self.g.adj_binaria[ui][vi] = 0

        custo, nova_rota = self.dijkstra(origem_lote, destino_lote)
        t_ms = (time.perf_counter() - t0) * 1000.0

        return {
            "Trecho_Travado": f"{u_falha} -> {v_falha}",
            "Nova_Rota_Ativa": " -> ".join(nova_rota) if nova_rota else "NENHUMA ROTA SEGURA (TRIP GERAL)",
            "Comprimento_Total": f"{custo:.1f} m" if custo < float('inf') else "∞",
            "Tempo_Decisao_ms": f"{t_ms:.4f}"
        }

# Execução do cenário de contingência
fms = criar_planta_fms_com_contingencia()
desviador = SistemaDesvioAutomaticoFMS(fms)

# Falha crítica: Travamento mecânico do trecho principal entre DIV-201 e DIV-202
resultado = desviador.tratar_evento_travamento("DIV-201", "DIV-202", "SILO-100", "CX-303")

print("=== RELATÓRIO SCADA: DESVIO AUTOMÁTICO DE PRODUÇÃO (FMS) ===")
print(formatar_tabela([resultado]))

# Asserções de conformidade
assert "DIV-203" in resultado["Nova_Rota_Ativa"]
assert "DIV-202" not in resultado["Nova_Rota_Ativa"]
assert float(resultado["Comprimento_Total"].split()[0]) < 10.0
print("\n[OK] Simulação de falha mecânica e reencaminhamento dinâmico validados com sucesso!")

=== RELATÓRIO SCADA: DESVIO AUTOMÁTICO DE PRODUÇÃO (FMS) ===
Trecho_Travado     | Nova_Rota_Ativa                           | Comprimento_Total | Tempo_Decisao_ms
-------------------+-------------------------------------------+-------------------+-----------------
DIV-201 -> DIV-202 | SILO-100 -> INSP-200 -> DIV-203 -> CX-303 | 3.8 m             | 0.0539          

[OK] Simulação de falha mecânica e reencaminhamento dinâmico validados com sucesso!
